In [ ]:
from langchain_openai import ChatOpenAI

API_KEY = "OPENROUTER_API_KEY"

llm = ChatOpenAI(
    model="openai/gpt-4.1-mini",
    base_url="https://openrouter.ai/api/v1",
    api_key=API_KEY,
    temperature=0,
    max_tokens=3000
)

In [ ]:
import os
import json
import textwrap
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display, Markdown


def extract_python_code(text):
    """
    Достает python-код из ответа LLM.
    """
    import re

    match = re.search(r"```python\s*(.*?)```", text, re.DOTALL)

    if match:
        return match.group(1).strip()

    return text.strip()


def run_data_description_agent(
    llm,
    train_path,
    target_column=None,
    business_task="",
    artifacts_dir="artifacts/data_description"
):
    """
    Data Description Agent:
    - анализирует только train;
    - не трогает test;
    - не делает preprocessing;
    - не обучает модели;
    - LLM сам пишет Python-код для EDA;
    - код выполняется локально;
    - результат сохраняется в artifacts.
    """

    Path(artifacts_dir).mkdir(parents=True, exist_ok=True)

    prompt = f"""
You are a Data Description Agent in a multi-agent ML system.

Analyze ONLY the training dataset.
Do NOT analyze test data.
Do NOT preprocess data.
Do NOT train models.
Do NOT drop or modify rows/columns.

You have to generate Python code that:
1. Loads the train dataset from:
   {train_path}

2. Detects:
   - dataset shape
   - columns and dtypes
   - target column
   - task type: regression, classification, or unknown
   - feature groups:
     numeric, categorical, text, datetime, boolean_like, id, target

3. Checks data quality:
   - missing values
   - duplicate rows
   - constant columns
   - high-cardinality columns
   - numeric outliers using IQR
   - target properties
   - class imbalance if classification

4. Saves these artifacts:
   - {artifacts_dir}/data_profile.json
   - {artifacts_dir}/schema.json
   - {artifacts_dir}/quality_report.json
   - {artifacts_dir}/data_description_report.md
   - {artifacts_dir}/data_description_summary.json

5. The final saved JSON must have this structure:
{{
  "agent": "data_description_agent",
  "status": "success",
  "skipped": false,
  "summary": "...",
  "decisions": {{
    "target_column": "...",
    "task_type": "...",
    "schema": {{
      "numeric": [],
      "categorical": [],
      "text": [],
      "datetime": [],
      "boolean_like": [],
      "id": [],
      "target": []
    }},
    "data_quality": {{}}
  }},
  "artifacts": {{
    "data_profile_path": "...",
    "schema_path": "...",
    "quality_report_path": "...",
    "data_description_report_path": "...",
    "data_description_summary_path": "..."
  }},
  "next_input": {{
    "target_column": "...",
    "task_type": "...",
    "schema": {{}},
    "data_summary": {{}}
  }},
  "reason": null
}}

Input:
- train_path: {train_path}
- target_column: {target_column}
- business_task: {business_task}
- artifacts_dir: {artifacts_dir}

Rules:
- Return ONLY Python code.
- No markdown explanation.
- The code must be executable in Jupyter.
- Use only pandas, numpy, json, pathlib.
- Do not use seaborn, matplotlib, sklearn, internet, APIs.
"""

    response = llm.invoke(prompt)
    code = extract_python_code(response.content)

    code_path = Path(artifacts_dir) / "generated_data_description_code.py"
    code_path.write_text(code, encoding="utf-8")

    namespace = {}
    exec(code, namespace)

    summary_path = Path(artifacts_dir) / "data_description_summary.json"

    with open(summary_path, "r", encoding="utf-8") as f:
        result = json.load(f)

    return result, code

In [ ]:
result, generated_code = run_data_description_agent(
    llm=llm,
    train_path="airbnb_train.csv",
    target_column=None,
    business_task="Predict Airbnb listing price"
)

result

In [ ]:
print("STATUS:", result["status"])
print("SUMMARY:", result["summary"])
print("TARGET:", result["decisions"]["target_column"])
print("TASK TYPE:", result["decisions"]["task_type"])

result["decisions"]["schema"]

In [ ]:
print(generated_code)